# Fail-closed allowlist cleaning

Create the APFS clone and apply only the documented, cell-equivalent CSV normalizations.

Documentation authority: [Zenodo record](https://zenodo.org/records/8089820), [Scientific Data descriptor](https://www.nature.com/articles/s41597-023-02445-z), and the publisher documentation retained through `data/raw/`. Unknown semantics always force preserve-only behavior.

## Paths and safety guards

The project is a sibling of the untouched `BCI Database/`. `data/raw/` is a read-only relative symlink to that source, while `data/processed/` is the derived APFS clone.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "data"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
required_directories = ("data", "models", "notebooks", "results")
if not all((PROJECT_ROOT / name).is_dir() for name in required_directories):
    raise RuntimeError("Run this notebook from bci_cleaning/ or its data/ or notebooks/ directory")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "results" / "reports"
LOGS_PATH = PROJECT_ROOT / "results" / "logs"
EXPECTED_RAW = PROJECT_ROOT.parent / "BCI Database"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != EXPECTED_RAW.resolve(strict=True):
    raise RuntimeError("data/raw must remain a relative symlink to the sibling BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)


## Phase implementation

In [2]:
"""Build a copy-on-write cleaned tree using only documentation-backed operations."""

from __future__ import annotations

import collections
import sys
from pathlib import Path

from bci_core import (
    ARTICLE_URL,
    MANIFEST_FIELDS,
    ZENODO_RECORD,
    archive_comparison_passed,
    assert_safe_paths,
    atomic_write_bytes,
    atomic_write_csv,
    atomic_write_text,
    clone_tree_apfs,
    common_parser,
    disk_free_bytes,
    is_administrative_artifact,
    log_event,
    new_run_id,
    normalize_frequency_csv_bytes,
    normalize_performances_csv_bytes,
    read_csv_dicts,
    sha256_file,
    verify_inventory_snapshot,
)


MIN_FREE_BYTES = 10 * 1024 * 1024 * 1024


def _blocking_preconditions(reports: Path) -> list[str]:
    errors: list[str] = []
    inventory = reports / "dataset_inventory.csv"
    comparison = reports / "archive_comparison.csv"
    issues = reports / "validation_issues.csv"
    if not inventory.exists():
        errors.append("dataset_inventory.csv is missing")
    if not archive_comparison_passed(comparison):
        errors.append("Zenodo archive comparison is missing or did not pass")
    if not issues.exists():
        errors.append("validation_issues.csv is missing")
    else:
        blocking = [row for row in read_csv_dicts(issues) if row.get("severity") in {"fatal", "error"}]
        if blocking:
            errors.append(f"validation_issues.csv contains {len(blocking)} blocking issue(s)")
    return errors


In [3]:
def main() -> int:
    parser = common_parser(__doc__ or "Clean dataset")
    args = parser.parse_args()
    raw, cleaned = assert_safe_paths(args.raw, args.cleaned)
    args.reports.mkdir(parents=True, exist_ok=True)
    args.logs.mkdir(parents=True, exist_ok=True)
    run_id = new_run_id()
    log_event(args.logs, run_id, "cleaning", "start", raw=str(raw), cleaned=str(cleaned))

    if not args.raw.is_symlink():
        raise RuntimeError("The project raw/ entry must remain a symlink to the untouched source dataset")
    preconditions = _blocking_preconditions(args.reports)
    if preconditions:
        for message in preconditions:
            print(f"BLOCKED: {message}", file=sys.stderr)
        log_event(args.logs, run_id, "cleaning", "blocked", reasons=preconditions)
        return 2
    if disk_free_bytes(cleaned.parent) < MIN_FREE_BYTES:
        raise RuntimeError("Less than 10 GiB is free; refusing to create even a copy-on-write clone")
    if cleaned.exists():
        destination_entries = {path.name for path in cleaned.iterdir()} if cleaned.is_dir() else set()
        if not cleaned.is_dir() or destination_entries - {".gitkeep"}:
            raise FileExistsError(f"Refusing nonempty cleaned destination: {cleaned}")

    inventory_path = args.reports / "dataset_inventory.csv"
    snapshot_errors = verify_inventory_snapshot(raw, inventory_path, args.logs, run_id)
    if snapshot_errors:
        for message in snapshot_errors:
            print(f"BLOCKED: {message}", file=sys.stderr)
        log_event(args.logs, run_id, "cleaning", "blocked", reasons=snapshot_errors)
        return 2

    clone_tree_apfs(raw, cleaned)
    inventory = read_csv_dicts(inventory_path)
    manifest: list[dict[str, str]] = []
    counts: collections.Counter[str] = collections.Counter()

    for row in inventory:
        relative = row["relative_path"]
        raw_path = raw / relative
        cleaned_path = cleaned / relative
        action = "preserved_byte_identical"
        reason = "No documentation-backed cleaning operation applies"
        source = ZENODO_RECORD
        equivalence = "expected-sha256-equality; pending 05_verify.ipynb"
        cleaned_hash = row["sha256"]

        if is_administrative_artifact(relative, raw_path):
            if cleaned_path.exists() or cleaned_path.is_symlink():
                cleaned_path.unlink()
            action = "excluded_administrative_artifact"
            reason = "Confirmed .DS_Store or invalid Office owner-lock file"
            equivalence = "raw-retained; cleaned-path-absent"
            cleaned_hash = ""
        elif cleaned_path.suffix.lower() == ".csv" and "frequency-band-selected" in cleaned_path.name:
            normalized, equivalence = normalize_frequency_csv_bytes(raw_path.read_bytes())
            atomic_write_bytes(cleaned_path, normalized)
            action = "normalized_frequency_csv"
            reason = "Removed the verified CRCRLF empty-line artifact and wrote UTF-8 LF records"
            source = ARTICLE_URL
            cleaned_hash = sha256_file(cleaned_path)
        elif cleaned_path.name == "Perfomances.csv":
            normalized, equivalence = normalize_performances_csv_bytes(raw_path.read_bytes())
            atomic_write_bytes(cleaned_path, normalized)
            action = "normalized_performance_csv"
            reason = "Losslessly transcoded CP1252 to UTF-8 and wrote LF record separators"
            source = ARTICLE_URL
            cleaned_hash = sha256_file(cleaned_path)

        counts[action] += 1
        manifest.append(
            {
                "raw_relative_path": relative,
                "cleaned_relative_path": "" if action.startswith("excluded_") else relative,
                "raw_sha256": row["sha256"],
                "cleaned_sha256": cleaned_hash,
                "action": action,
                "reason": reason,
                "documentation_source": source,
                "equivalence_check": equivalence,
                "status": "complete",
            }
        )

    atomic_write_csv(args.reports / "cleaning_manifest.csv", manifest, MANIFEST_FIELDS)
    report_lines = [
        "# Cleaning Report",
        "",
        f"- Run ID: `{run_id}`",
        f"- Raw source: `{raw}`",
        f"- Cleaned tree: `{cleaned}`",
        f"- Raw files represented: {len(manifest):,}",
        "- Raw dataset modified: no",
        "- GDF recordings modified: no",
        "- Participant IDs, timestamps, runs, and trial order modified: no",
        "",
        "## Actions",
        "",
    ]
    for action, count in sorted(counts.items()):
        report_lines.append(f"- `{action}`: {count:,} files")
    report_lines.extend(
        [
            "",
            "## Safeguards",
            "",
            "- Cleaning ran only after the Zenodo path/size/CRC32 comparison passed.",
            "- The raw SHA-256 snapshot was rechecked immediately before cloning.",
            "- CSV changes were written atomically and accepted only after parsed-cell equivalence checks.",
            "- All unknown or scientifically meaningful file types were preserved byte-for-byte.",
            "- Final independent hashing remains the responsibility of `05_verify.ipynb`.",
            "",
            "## Unresolved issues",
            "",
            "See `validation_issues.csv`. Warnings remain unchanged in the dataset and are not silently corrected.",
            "",
        ]
    )
    atomic_write_text(args.reports / "cleaning_report.md", "\n".join(report_lines))
    log_event(args.logs, run_id, "cleaning", "complete", files=len(manifest), actions=dict(counts))
    print(f"Cleaning complete: {len(manifest)} raw files represented")
    print("Run 05_verify.ipynb before treating the cleaned dataset as complete.")
    return 0


## Execute phase

In [4]:
_arguments = [
    "data_cleaning.ipynb",
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--logs", str(LOGS_PATH),
]

_original_argv = sys.argv[:]
try:
    sys.argv = _arguments
    _exit_code = main()
finally:
    sys.argv = _original_argv

if _exit_code != 0:
    raise RuntimeError(f"Phase returned nonzero status {_exit_code}; inspect validation_issues.csv")


Cleaning complete: 1073 raw files represented
Run 05_verify.ipynb before treating the cleaned dataset as complete.


## Privacy-safe report check

In [5]:
import csv

report_path = REPORTS_PATH / "cleaning_manifest.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "rows": len(report_rows),
}


{'report': 'results/reports/cleaning_manifest.csv', 'rows': 1073}